In [ ]:
## install openai library
!pip install openai

In [ ]:
## import necessary libraries
import os
from openai import OpenAI
import openai
import json

In [ ]:
## Mount colab
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

In [ ]:
# Activate API key
client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

In [ ]:
import json
import pandas as pd

with open("/content/drive/MyDrive/MYPROJECTFOLDER/QSQs.json", "r") as f:
    data = json.load(f)

QSQs_data = pd.DataFrame(
    list(data.items()),
    columns=["Qid", "Q"]
)

QSQs_data.head()

In [ ]:
## let's look at the unique Qids:
QSQs_data.Qid.value_counts()

In [ ]:
## let's look at the unique texts:
QSQs_data.Q.value_counts()

In [ ]:
## let's look at an example of the text
QSQs_data.Q.iloc[65]

In [ ]:
## create function to return API message content
def gpt_response(prompt, input_text):
    completion = client.chat.completions.create(
      model="gpt-4.1-mini",
      messages=[
        {"role": "system", "content": prompt},
        {"role": "user", "content": input_text}
        ]
      )
    return completion.choices[0].message.content

In [ ]:
# defensive API calling
import time

def gpt_response(prompt, input_text, retries=3, delay=3):
    """
    Robust GPT API request: retries if it fails, always returns a string,
    and logs the error if unsuccessful.
    """
    for attempt in range(1, retries + 1):
        try:
            completion = client.chat.completions.create(
                model="gpt-4.1-mini",
                messages=[
                    {"role": "system", "content": prompt},
                    {"role": "user", "content": input_text}
                ]
            )
            response_text = completion.choices[0].message.content
            if response_text is None or not response_text.strip():
                raise ValueError("Empty response from API")
            return response_text
        except Exception as e:
            print(f"GPT call failed on attempt {attempt}: {e}")
            time.sleep(delay)  # wait before retrying

    # All retries failed, return a placeholder
    print(f"GPT call ultimately failed after {retries} attempts for input: {input_text[:80]}")
    return ""  # Or you might return "{}" to indicate empty JSON

In [ ]:
## test function on example Q
prompt = """CONTEXT: You will be given a question posed by a Member of Parliament (MP)
    during question time in Zimbabwe's National Assembly (lower house).
    The question, asked between September 2015 and August 2023, is addressed to a government minister.
    ###
    TASK: Extract information from the question to classify it according to the following variables.
    Pay close attention to both explicit and implicit details.
    Always consider the broader context rather than isolated keywords.
    ###
    IMPORTANT CODING RULE:
    For any variable whose instructions specify coding as "NA" unless a condition is met,
    you MUST return "NA" (not 0 or 1) if the required conditions are NOT met.
    ###
    VARIABLES TO CLASSIFY
    1.	nature (Question Type)
        o	Classify whether the MP:
            -	requests a service (e.g., construction, repair, provision), or
            -	inquires about government policy on a specific issue or portfolio area.
        o	IMPORTANT: MPs sometimes phrase service requests as policy questions
            to avoid their question being ruled out by the Speaker.
            Always interpret the question's intent in context.
        o	IMPORTANT: MPs sometimes frame questions as points of order OR privilege
            to immediately get the floor for asking follow-up questions.
            Always interpret the question's intent in context.
        o	If uncertain, code as "DK" with a brief rationale.

    2.	transport (Service Related to Transport)
        o	If nature = service, classify whether the request relates broadly
            to transport infrastructure or transport safety.
        o	Code as 1 if yes, otherwise 0.
        o	If nature ≠ service, CODE AS "NA".

    3.	road (Road Infrastructure Request)
        o	If nature = service AND transport = 1, classify if the request involves:
            -	new road construction, restoration of damaged roads, or continuation of interrupted work.
        o	Code as 1 if yes, otherwise 0.
        o	If nature = service BUT transport ≠ 1, CODE AS "NA".
        o	If nature ≠ service, CODE AS "NA".

    4.	education (Service Related to Education)
        o	If nature = service, classify whether the request relates to primary, secondary, or tertiary education.
        o	Code as 1 if yes, otherwise 0.
        o	If nature ≠ service, CODE AS "NA".

    5.	school (School Infrastructure Request)
        o	If nature = service AND education = 1, classify if the request includes:
            -	new school construction, restoration of damaged schools, or resumption of halted work.
        o	Code as 1 if yes, otherwise 0.
        o	If nature = service BUT education ≠ 1, CODE AS "NA".
        o	If nature ≠ service, CODE AS "NA".

    6.	health (Service Related to Health)
        o	If nature = service, classify if the request pertains to health services.
        o	Code as 1 if yes, otherwise 0.
        o	If nature ≠ service, CODE AS "NA".

    7.	clinic (Clinic/Hospital Infrastructure Request)
        o	If nature = service AND health = 1, classify if the request includes:
            -	new clinic/hospital construction, restoration, or resumption of halted work.
        o	Code as 1 if yes, otherwise 0.
        o	If nature = service BUT health ≠ 1, CODE AS "NA".
        o	If nature ≠ service, CODE AS "NA".

    8.	water (Service Related to Water)
        o	If nature = service, classify if the request relates to water resources management or supply.
        o	Code as 1 if yes, otherwise 0.
        o	If nature ≠ service, CODE AS "NA".

    9.	well (Borehole Request)
        o	If nature = service AND water = 1, classify if the request involves:
            -	drilling new boreholes, restoring dilapidated boreholes, or resuming interrupted borehole work.
        o	Code as 1 if yes, otherwise 0.
        o	If nature = service BUT water ≠ 1, CODE AS "NA".
        o	If nature ≠ service, CODE AS "NA".

    10.	regime (Serious Government Concern)
        o	Classify whether the question raises concerns related to:
            -	human rights violations
                +	civil liberty restrictions
                +	personal integrity violations
            -	electoral integrity issues
                +	election-day fraud
                +	politicized issuance of identity documents
                +	unfair voter registration
                +	voter intimidation
                +	biased information environment
                +	politicization of the electoral commission
            -	rule of law and democratic norms violations
                +	political interference
                +	compromised separation of powers
                +	compromised horizontal accountability
                +	lack of political forebearance
                +	property rights violations
                +	arbitrary exercise of power
            -	corruption/embezzlement
            -	political violence/discrimination
            -	political instability within ZANU(PF)
                +	infighting between Lacoste and G40 factions
                +	internal power struggles
        o	Code as 1 if any apply, otherwise 0.
        o	IMPORTANT: code as 1 ONLY IF there are clear implicit or explicit
            indications that an MP expresses any such concern.
        o	IMPORTANT: MPs who raise such concerns often frame
            their questions in suggestive ways.
        o	For ambiguous cases, code "DK" with a brief rationale.

    11.	scope (Geographical Scope)
        o	Classify whether the question concerns:
            -	local issues,
            - regional issues, or
            - national issues.
        o	If unclear, code “DK” with a brief rationale.

    12.	critic (Blame Attribution)
        o	Classify whether the MP points blame explicitly or implicitly at:
            -	the government ("govt"),
            -	other forces ("other"), or
            -	makes no blame attribution ("none").
        o	IMPORTANT: the government ("govt") EXCLUSIVELY means the national government
            or national government ministries. If blamed, local authorities
            and MPs MUST be coded as other forces ("other").
        o	IMPORTANT: Always DOUBLE CHECK whether the government is INDEED implicitly
            or explicitly blamed for an issue. Blame is NOT NECESSARILY attributed to the government
            if it is just asked to clarify how it seeks to deal with an issue.
        o	IMPORTANT: MPs sometimes attribute blame in suggestive ways,
            notably when blaming the government for an issue.
        o	If unclear, code “DK” with a brief rationale.

    ###
    FIRST IMPORTANT CODING RULE:
    If nature = service, these variables MUST always be classified as 0 or 1:
        o	transport
        o	education
        o	health
        o	water
    ###
    SECOND IMPORTANT CODING RULE:
    If you INCORRECTLY use "0" instead of "NA" WHERE A CONDITION REQUIRES "NA",
    this will be counted as an error.

    THIRD IMPORTANT CODING RULE:
        o	For any variable with "DK" as a possible code:
            -	If you code "DK", ALWAYS append a brief rationale after a comma.
            - If you append a brief rationale, NEVER use quotation marks when referring to some parts of the question.
            -	If you select any other allowed value, output ONLY the value WITHOUT RATIONALE.

    ###
    OUTPUT FORMAT
    Return VALID JSON ONLY with the structure below.
    Replace variable values as per your classification:
    {
    "nature": "service | policy | DK, [brief rationale]",
    "transport": 1 | 0 | "NA",
    "road": 1 | 0 | "NA",
    "education": 1 | 0 | "NA",
    "school": 1 | 0 | "NA",
    "health": 1 | 0 | "NA",
    "clinic": 1 | 0 | "NA",
    "water": 1 | 0 | "NA",
    "well": 1 | 0 | "NA",
    "regime": 1 | 0 | "DK, [brief rationale]",
    "scope": "local | regional | national | DK, [brief rationale]",
    "critic": "govt | other | none | DK, [brief rationale]"
    }
    ###
    FINAL CHECK: Output a brief rationale ONLY AFTER "DK", and NEVER after values that are NOT "DK".
    """

gpt_response(prompt, QSQs_data.Q.iloc[65])

In [ ]:
## optional -- just to see the loop progress
from tqdm import tqdm

## now run function on all entries of our JSON file and store responses
gpt_responses = []

for message in tqdm(QSQs_data.Q):
  response = gpt_response(prompt, message)
  gpt_responses.append(response)

In [ ]:
QSQs_data["gpt_response"] = gpt_responses

In [ ]:
 # Ultimate fix to have a returning loop that catches all failed Qs (up till a max is reached)

def is_valid_json(s):
    try:
        if not isinstance(s, str):
            return False
        s = s.strip()
        if not s.startswith("{"):
            return False
        import json
        json.loads(s)
        return True
    except Exception:
        return False

def gpt_response(prompt, input_text, retries=3, delay=3):
    # ... (your robust function here)
    completion = client.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": prompt},
            {"role": "user", "content": input_text},
        ]
    )
    return completion.choices[0].message.content or ""

def rerun_missing(df, prompt, response_col="gpt_response", max_loops=5, save_path_base=None):
    """
    Loops through the dataframe rerunning GPT on rows with invalid JSON responses
    until all are valid, or max_loops is reached.
    """
    df = df.copy()
    loop = 0
    while True:
        # Mask of invalid responses
        failed_mask = ~df[response_col].apply(is_valid_json)

        # If none failed, we're done
        num_failed = failed_mask.sum()
        print(f"Rerun round {loop}: {num_failed} failed responses to rerun.")
        if num_failed == 0:
            print("All responses valid JSON.")
            break
        if loop >= max_loops:
            print("Maximum rerun loops reached.")
            break

        # Optionally save failed for inspection
        if save_path_base:
            failed_path = f"{save_path_base}_failed_round{loop}.json"
            df[failed_mask][["Qid", "Q", response_col]].to_json(
                failed_path, orient="records", force_ascii=False, indent=2
            )
            print(f"Saved failed responses for round {loop} to {failed_path}")

        # Extract failed Q's
        failed_qs = df.loc[failed_mask, ["Qid", "Q"]]

        print(f"Rerunning GPT for {len(failed_qs)} failed responses...")
        rerun_gpt_responses = []
        for msg in tqdm(failed_qs["Q"]):
            response = gpt_response(prompt, msg)
            rerun_gpt_responses.append(response)
            time.sleep(2)  # or adjust as needed
        rerun_failed = failed_qs.copy()
        rerun_failed[response_col] = rerun_gpt_responses

        # Update the main dataframe
        rerun_response_map = dict(zip(rerun_failed["Qid"], rerun_failed[response_col]))
        def updated_response(row):
            if is_valid_json(row[response_col]):
                return row[response_col]
            return rerun_response_map.get(row["Qid"], row[response_col])
        df[response_col] = df.apply(updated_response, axis=1)

        loop += 1

    return df

final_df = rerun_missing(
    QSQs_data,
    prompt,
    response_col="gpt_response",
    max_loops=5,  # prevent infinite loops
    save_path_base="/content/drive/MyDrive/MYPROJECTFOLDER/QSQs_failed_reruns"
)

# Save final output!
output_path = "/content/drive/MyDrive/MYPROJECTFOLDER/QSQs_gpt_Nat_final.json"
final_df.to_json(output_path, orient="records", force_ascii=False, indent=2)

In [ ]:
# Strip markdown code fencing before json.loads

import re

def strip_markdown_fence(text):
    """
    Removes markdown code fencing from a string.
    Accepts strings like '```json\n{...}\n```' or '```\n{...}\n```', returns the {...} as a string.
    """
    if isinstance(text, str):
        # Remove ```json ... ``` or ``` ... ```
        text = re.sub(r"^```(?:json)?\s*", "", text.strip(), flags=re.IGNORECASE)
        text = re.sub(r"\s*```$", "", text, flags=re.IGNORECASE)
    return text

def safe_json_loads(s):
    try:
        s = strip_markdown_fence(s)
        return json.loads(s)
    except Exception:
        return None  # Or raise, or return {}, etc.

In [ ]:
# Turn the gpt_response column into actual dictionnaries
QSQs_data["gpt_response_dict"] = QSQs_data["gpt_response"].apply(safe_json_loads)

In [ ]:
# Expand the classification variables into individual columns
gpt_class_df = QSQs_data["gpt_response_dict"].apply(pd.Series)

In [ ]:
# Concatenate
final_df = pd.concat([QSQs_data[["Qid", "Q"]], gpt_class_df], axis=1)

In [ ]:
final_df

In [ ]:
output_path = "/content/drive/MyDrive/MYPROJECTFOLDER/QSQs_gpt_NoShot.json"

final_df.to_json(output_path, orient="records", force_ascii=False, indent=2)
